# tiny-voca in Luganda: two voices, one model trained on both T4s

Fine-tunes the English [tiny-voca](https://github.com/n4ze3m/tiny-voca) (4.5M parameters) on
`CraneAILabs/waxal-lug-clean` with one female voice (F2) and one male voice (M2).
Training is data-parallel: both GPUs train the same model, each on its own batches.

**Everything lives in `/kaggle/working/`, and every step resumes.** A finished step leaves a marker in
`/kaggle/working/stages/` and is skipped next time; an interrupted step continues where it stopped
(training from its last save, at most 500 steps back). To pick up a stopped run:

- **Interactive session:** Settings → *Persistence* → **Files only**, then *Run All* again.
- **Saved version:** *Add Input* → your previous version's output, set `RESUME_FROM` in cell 1, then *Save & Run All*.

Settings: Accelerator **GPU T4 ×2**, Internet **on**.

| Step | Cell | Uses | Time |
|---|---|---|---|
| 1. Setup and downloads | 1–6 | | ~5 min |
| 2. Export the two speakers' audio at 24 kHz | 7 | CPU | ~5 min |
| 3. Cut long clips into 4–18 s chunks | 8 | both GPUs, one speaker each | ~5 min |
| 4. Acoustic features | 9 | all CPU cores | ~15–25 min |
| 5. Phoneme alignment | 10 | GPU 0 | ~5 min |
| 6. Training: 4,000 steps | 11 | both GPUs, one model | cell 11 prints the ETA |
| 7. Listen to checkpoints, package | 12–13 | | ~10 min |

In [1]:
# 1. Settings and folders
import os, sys, json, time, shutil, subprocess, hashlib, re
from pathlib import Path

FEMALE, MALE = "5", "1"          # Waxal speaker_id: 5 (female) and 1 (male) have the most audio
MINUTES_PER_VOICE = 100          # training audio per voice after cutting
STEPS = 4000                     # one step = one batch per GPU; about 80 passes over the data
LEARNING_RATE = 2e-4             # doubled for the doubled batch (two GPUs)
BATCH_FRAMES = 9000              # per GPU; drops to 6000 automatically if a T4 runs out of memory
MAX_TRAIN_SECONDS = 3 * 3600     # per session; training resumes from its last save next time
RESUME_FROM = None               # e.g. "/kaggle/input/tiny-voca-luganda" (a previous version's output)

WORK = Path("/kaggle/working")
REPO = WORK / "tiny-voca"        # code
OUT = WORK / "luganda"           # scripts, samples, final model
DATA = WORK / "data"             # clips, chunks, manifests
FEATURES = WORK / "features"     # acoustic features
ANN = WORK / "annotations"       # phonemes and durations
RUN = WORK / "run"               # training checkpoints
LOGS = WORK / "logs"
STAGES = WORK / "stages"         # one marker per finished step
voices = {FEMALE: "F2", MALE: "M2"}

if RESUME_FROM and not any(STAGES.glob("*.done")):
    print(f"copying the previous run from {RESUME_FROM} ...")
    shutil.copytree(RESUME_FROM, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__notebook__*", "*.html"))
for d in (OUT, DATA, FEATURES, LOGS, STAGES):
    d.mkdir(parents=True, exist_ok=True)

# The data steps depend on these settings: refuse to mix runs made with different ones.
settings = {"female": FEMALE, "male": MALE, "minutes_per_voice": MINUTES_PER_VOICE}
settings_file = STAGES / "settings.json"
if settings_file.exists() and json.loads(settings_file.read_text()) != settings:
    raise SystemExit(f"Settings changed since the saved run ({settings_file.read_text()}). "
                     "Delete /kaggle/working/stages, data, features, annotations and run to start over.")
settings_file.write_text(json.dumps(settings))

os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["TINY_VOCA_FRONTEND"] = "lug"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def done(stage):
    return (STAGES / f"{stage}.done").exists()

def mark(stage, **info):
    (STAGES / f"{stage}.done").write_text(json.dumps(info) + "\n")
    print(f"✓ {stage} finished: {info}")

def skip(stage):
    if done(stage):
        print(f"✓ {stage}: already done, skipping ({(STAGES / f'{stage}.done').read_text().strip()})")
        return True
    return False

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def write_jsonl(path, rows):
    Path(path).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")

def last_line(text):
    lines = [l for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else ""

def start(name, cmd, gpu=None, single_thread=False):
    log = LOGS / f"{name}.log"
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    if single_thread:  # one job per core: stop numpy/numba/torch from oversubscribing the CPU
        env.update({k: "1" for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMBA_NUM_THREADS")})
    return subprocess.Popen(cmd, env=env, stdout=open(log, "a"), stderr=subprocess.STDOUT), log

def watch(jobs, every=60):
    # jobs: {name: (Popen, log path)}; prints progress, raises on failure
    while True:
        running = False
        for name, (proc, log) in jobs.items():
            text = Path(log).read_text(errors="ignore")
            code = proc.poll()
            if code is None:
                running = True
                print(f"[{(time.time() - T0) / 60:6.1f} min] {name}: {last_line(text)[:160] or 'working (prints when done)'}", flush=True)
            elif code != 0:
                print(text[-4000:])
                raise RuntimeError(f"{name} failed with exit code {code}; rerun this cell to resume")
        if not running:
            for name, (proc, log) in jobs.items():
                print(f"{name}: {last_line(Path(log).read_text(errors='ignore'))[:300]}")
            return
        time.sleep(every)

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus))
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
print("finished steps:", sorted(p.stem for p in STAGES.glob("*.done")) or "none")
T0 = time.time()

GPU 0: Tesla T4 (UUID: GPU-c83cded5-1d00-e9a2-c75f-01789630634c)
GPU 1: Tesla T4 (UUID: GPU-f8ceb36f-bf25-9f62-ee23-dd57c7b7d18f)
finished steps: none


In [2]:
# 2. Code, weights and the phoneme aligner (installed every session; downloads are kept)
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/n4ze3m/tiny-voca", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "librosa", "soundfile", "pyarrow", "scipy"], check=True)
sys.path.insert(0, str(REPO / "src"))

from huggingface_hub import hf_hub_download, snapshot_download
WEIGHTS = WORK / "weights" / "tiny-voca.pt"
if not WEIGHTS.exists():
    WEIGHTS = Path(hf_hub_download("n4ze3m/tiny-voca", "tiny-voca.pt", local_dir=WORK / "weights"))
if not (done("split") and done("annotate")):
    snapshot_download("facebook/wav2vec2-lv-60-espeak-cv-ft")   # used only to align training data
print("English weights:", WEIGHTS)

tiny-voca.pt: reconstructing file:   0%|          |  0.00B / 18.2MB            

tiny-voca.pt: downloading bytes:           |  0.00B            

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

English weights: /kaggle/working/weights/tiny-voca.pt


In [3]:
%%writefile /kaggle/working/luganda/lug.py
"""Luganda front end: spelling -> tiny-voca inventory phones.

Luganda spelling is close to phonemic, so plain rules are enough:
* a double vowel is a long vowel (aa -> ɑː, ee -> eɪ, ii -> iː, oo -> oː, uu -> uː);
* a double consonant is a geminate and is spoken as the phone twice (kk -> k k);
* ny is the palatal nasal (n j), ng' (or ŋ) is the velar nasal, ky/gy are palatal stops;
* other apostrophes mark elision (n'omwana) and are dropped.

Every output phone is already in the English inventory, so the English weights
and the phoneme aligner can be reused unchanged. Tone is not written in Luganda
spelling and is not modelled. Digits are not read: spell numbers out first.
"""

import re
import unicodedata

VERSION = "lug-rules-1"

VOWELS = {"a": ("ɐ", "ɑː"), "e": ("ɛ", "eɪ"), "i": ("i", "iː"), "o": ("ɔ", "oː"), "u": ("ʊ", "uː")}
DIGRAPHS = {"ng'": ("ŋ",), "ny": ("n", "j"), "ky": ("tʃ",), "gy": ("dʒ",), "sh": ("ʃ",), "ch": ("tʃ",)}
LETTERS = {"b": ("b",), "c": ("tʃ",), "d": ("d",), "f": ("f",), "g": ("ɡ",), "h": ("h",), "j": ("dʒ",),
           "k": ("k",), "l": ("l",), "m": ("m",), "n": ("n",), "p": ("p",), "q": ("k",), "r": ("ɾ",),
           "s": ("s",), "t": ("t",), "v": ("v",), "w": ("w",), "x": ("k", "s"), "y": ("j",), "z": ("z",),
           "ŋ": ("ŋ",)}
_WORD = re.compile(r"[a-zŋ']+")


def identity():
    """Everything that decides the output, for the front-end fingerprint."""
    return {"version": VERSION, "vowels": VOWELS, "digraphs": DIGRAPHS, "letters": LETTERS}


def clean(text):
    text = unicodedata.normalize("NFC", text).lower().replace("’", "'").replace("ʼ", "'").replace("‘", "'")
    # Strip accents (é -> e) but keep ŋ, which has no decomposition.
    text = "".join(c for c in unicodedata.normalize("NFD", text) if not unicodedata.combining(c))
    return text.replace("ng'", "ŋ")


def word_phones(word):
    phones, i = [], 0
    while i < len(word):
        char = word[i]
        if char == "'":
            i += 1
            continue
        if char in VOWELS:
            long = i + 1 < len(word) and word[i + 1] == char
            phones.append(VOWELS[char][1 if long else 0])
            i += 2 if long else 1
            continue
        pair = word[i:i + 2]
        # Geminate digraph: kky, ggy, nny -> the digraph twice.
        if i + 2 < len(word) and word[i + 1] == char and word[i + 1:i + 3] in DIGRAPHS:
            phones.extend(DIGRAPHS[word[i + 1:i + 3]] * 2 if word[i + 1:i + 3] != "ny" else ("n", "n", "j"))
            i += 3
            continue
        if pair in DIGRAPHS:
            phones.extend(DIGRAPHS[pair])
            i += 2
            continue
        if char in LETTERS:
            phones.extend(LETTERS[char])
        i += 1  # anything else (digits, symbols) is skipped
    return phones


def words(piece):
    """One text piece (no sentence punctuation) -> words as lists of (phone, stress)."""
    result = []
    for word in _WORD.findall(clean(piece)):
        phones = word_phones(word)
        if phones:
            result.append([(phone, 0) for phone in phones])
    return result

Writing /kaggle/working/luganda/lug.py


In [4]:
%%writefile /kaggle/working/luganda/apply_patch.py
"""Add the Luganda front end ("lug") and --init-weights to a tiny-voca checkout.

Usage: python apply_patch.py /path/to/tiny-voca /path/to/lug.py
Safe to run twice: a file that is already patched is left alone.
"""

import shutil
import sys
from pathlib import Path

repo, lug = Path(sys.argv[1]), Path(sys.argv[2])
package = repo / "src" / "tiny_voca"
shutil.copy(lug, package / "lug.py")


def patch(path, edits, marker):
    text = path.read_text(encoding="utf-8")
    if marker in text:
        print(f"{path.name}: already patched")
        return
    for old, new in edits:
        if text.count(old) != 1:
            raise SystemExit(f"{path.name}: expected exactly one match for {old[:60]!r}")
        text = text.replace(old, new)
    path.write_text(text, encoding="utf-8")
    print(f"{path.name}: patched")


# "lug" goes before "own" so that checkpoint matching never needs dizione.
patch(package / "text.py", [
    ('FRONTENDS = ("espeak", "own")', 'FRONTENDS = ("espeak", "lug", "own")'),
    ('def frontend_identity():\n',
     'def frontend_identity():\n'
     '    if _frontend == "lug":  # LUGANDA-PATCH\n'
     '        from . import lug\n'
     '        return {"frontend_version": lug.VERSION, "phonemizer": "lug-rules", "symbols": list(SYMBOLS),\n'
     '                "rules": json.loads(json.dumps(lug.identity(), ensure_ascii=False))}\n'),
    ('def frontend_runtime():\n    """What an annotation run records about the exact program behind its\n'
     '    tokens: the eSpeak version, or the fingerprint of the phonemizer build."""\n',
     'def frontend_runtime():\n    """What an annotation run records about the exact program behind its\n'
     '    tokens: the eSpeak version, or the fingerprint of the phonemizer build."""\n'
     '    if _frontend == "lug":\n'
     '        from . import lug\n'
     '        return {"phonemizer": lug.VERSION}\n'),
    ('    if _frontend == "own":\n        return tokenize(_dizione().pieces(text))\n',
     '    if _frontend == "own":\n        return tokenize(_dizione().pieces(text))\n'
     '    if _frontend == "lug":\n'
     '        from . import lug\n'
     '        return tokenize([(True, piece) if punctuation else (False, lug.words(piece))\n'
     '                         for punctuation, piece in chunks(normalize_text(text))])\n'),
], marker="LUGANDA-PATCH")

# Start a run from the released inference weights (they hold no critics or
# optimizer state, so --init-from cannot use them).
patch(package / "train.py", [
    ('    parser.add_argument("--stop-file"',
     '    parser.add_argument("--init-weights", type=Path, help="Start a NEW run from an inference checkpoint '
     '(such as the released tiny-voca.pt): speech-model weights only; critics start fresh")  # INIT-WEIGHTS-PATCH\n'
     '    parser.add_argument("--stop-file"'),
    ('    recipe["branched_from"] = None\n    step = 0\n',
     '    recipe["branched_from"] = None\n    step = 0\n'
     '    if args.init_weights:\n'
     '        if args.init_from:\n'
     '            parser.error("--init-weights and --init-from both choose the starting weights")\n'
     '        weights = torch.load(args.init_weights, map_location="cpu", weights_only=True)\n'
     '        if weights["config"] != c.to_dict():\n'
     '            raise ValueError("--init-weights: the checkpoint has a different model configuration")\n'
     '        recipe["init_weights"] = {"path": str(args.init_weights), "step": weights.get("step")}\n'
     '        if not args.resume:  # with --resume, pass the same --init-weights so the recipe matches\n'
     '            system.model.load_state_dict(weights["model"], strict=True)\n'),
], marker="INIT-WEIGHTS-PATCH")

# Data-parallel training on several GPUs (torchrun --nproc_per_node=N -m tiny_voca.train ...).
# Each process trains on its own batches; gradients are averaged before every
# optimizer step, so all processes keep one identical model. Only rank 0 logs,
# validates and saves.
patch(package / "train.py", [
    ('import torch\nfrom torch import nn\nfrom torch.nn import functional as F\n',
     'import torch\nfrom torch import nn\nfrom torch.nn import functional as F\n'
     'from datetime import timedelta\n'
     'import tempfile\n'
     'import torch.distributed as dist\n\n\n'
     'def world_size():  # DDP-PATCH\n'
     '    return dist.get_world_size() if dist.is_available() and dist.is_initialized() else 1\n\n\n'
     'def average_gradients(module):\n'
     '    """Average gradients over all processes before an optimizer step."""\n'
     '    world = world_size()\n'
     '    params = [p for p in module.parameters() if p.requires_grad]\n'
     '    if world == 1 or not params:\n'
     '        return\n'
     '    present = torch.tensor([float(p.grad is not None) for p in params], device=params[0].device)\n'
     '    flat = torch.cat([(p.grad if p.grad is not None else torch.zeros_like(p)).reshape(-1).float() for p in params] + [present])\n'
     '    dist.all_reduce(flat)\n'
     '    flat /= world\n'
     '    offset = 0\n'
     '    for p, keep in zip(params, (flat[-len(params):] > 0).tolist()):\n'
     '        n = p.numel()\n'
     '        p.grad = flat[offset:offset + n].view_as(p).to(p.dtype).clone() if keep else None\n'
     '        offset += n\n'),
    ('        d_loss.backward()\n',
     '        d_loss.backward()\n        average_gradients(critics)\n'),
    ('    grad = nn.utils.clip_grad_norm_(system.parameters()',
     '    average_gradients(system)\n    grad = nn.utils.clip_grad_norm_(system.parameters()'),
    ('"frames": int(batch["frame_lengths"].sum()), "utterances": len(batch["frame_lengths"])}',
     '"frames": int(batch["frame_lengths"].sum()) * world_size(), "utterances": len(batch["frame_lengths"]) * world_size()}'),
    ('    c = Config.from_dict(json.loads(args.config.read_text())) if args.config else Config()\n    torch.manual_seed(args.seed)\n',
     '    world, rank = int(os.environ.get("WORLD_SIZE", "1")), int(os.environ.get("RANK", "0"))\n'
     '    if world > 1:\n'
     '        if str(args.device).startswith("cuda"):\n'
     '            torch.cuda.set_device(int(os.environ.get("LOCAL_RANK", "0")))\n'
     '            args.device = f"cuda:{torch.cuda.current_device()}"\n'
     '        dist.init_process_group("nccl" if str(args.device).startswith("cuda") else "gloo", timeout=timedelta(minutes=30))\n'
     '        if rank != 0:  # only rank 0 writes into --output\n'
     '            args.output = Path(tempfile.mkdtemp(prefix=f"tiny-voca-rank{rank}-"))\n'
     '    c = Config.from_dict(json.loads(args.config.read_text())) if args.config else Config()\n    torch.manual_seed(args.seed)\n'),
    ('args.batch_frames, args.max_batch, args.seed)',
     'args.batch_frames, args.max_batch, args.seed + rank)'),
    ('        sampler.load_state(state["sampler"])\n',
     '        sampler.load_state(state["sampler"])\n'
     '        if rank != 0:  # rank 0 saved its own batch order; the others draw a fresh one\n'
     '            sampler = BucketSampler([train.lengths[i] for i in pool], args.batch_frames, args.max_batch, args.seed + rank + state["step"])\n'),
    ('    if step >= args.steps:\n',
     '    if world > 1:  # start every process from rank 0\'s exact weights\n'
     '        for tensor in list(system.state_dict().values()) + list(critics.state_dict().values()):\n'
     '            dist.broadcast(tensor, 0)\n'
     '    if step >= args.steps:\n'),
    ('        retain = step == args.gan_start or step % args.keep_every == 0\n',
     '        if world > 1:  # stop together, or the others wait forever\n'
     '            flag = torch.tensor([float(should_stop)], device=args.device)\n'
     '            dist.all_reduce(flag, op=dist.ReduceOp.MAX)\n'
     '            should_stop = bool(flag.item())\n'
     '        retain = step == args.gan_start or step % args.keep_every == 0\n'),
    ('        if step % args.log_every == 0 or save:\n',
     '        if rank == 0 and (step % args.log_every == 0 or save):\n'),
    ('        if should_stop:\n            break\n',
     '        if should_stop:\n            break\n'
     '    if world > 1:\n        dist.barrier()\n        dist.destroy_process_group()\n'),
], marker="DDP-PATCH")

# Annotation: a train+validation run is complete (upstream also requires a test
# split), and Luganda vowels are aligned against the recogniser's plain vowel
# labels (a, e, o, u) when it has them, which it scores far more confidently
# than the English-flavoured inventory vowels.
patch(package / "annotate.py", [
    ('    complete = not args.limit and set(args.splits) == set(SPLITS)\n',
     '    complete = not args.limit  # ANNOTATE-PATCH: train+validation without a test split is complete\n'),
], marker="ANNOTATE-PATCH")
patch(package / "align.py", [
    ('    def label(self, symbol):\n'
     '        symbol = symbol if symbol in self.vocabulary else ALIGNER_LABEL_FALLBACK[symbol]\n',
     '    def label(self, symbol):\n'
     '        from .text import frontend  # ALIGN-PATCH\n'
     '        if frontend() == "lug":\n'
     '            preferred = LUGANDA_LABELS.get(symbol)\n'
     '            if preferred in self.vocabulary:\n'
     '                symbol = preferred\n'
     '        symbol = symbol if symbol in self.vocabulary else ALIGNER_LABEL_FALLBACK[symbol]\n'),
    ('ALIGNER_LABEL_FALLBACK = {UNK: "<unk>"}\n',
     'ALIGNER_LABEL_FALLBACK = {UNK: "<unk>"}\n'
     '# Recogniser labels closer to Luganda sounds, used for alignment only (tokens are unchanged).\n'
     'LUGANDA_LABELS = {"ɐ": "a", "ɑː": "aː", "ɛ": "e", "eɪ": "eː", "ɔ": "o", "oː": "oː", "ʊ": "u", "uː": "uː", "iː": "iː"}\n'),
], marker="ALIGN-PATCH")

Writing /kaggle/working/luganda/apply_patch.py


In [5]:
%%writefile /kaggle/working/luganda/split_clips.py
"""Cut long recordings into 4-18 s chunks at pauses between words.

tiny-voca accepts at most 32 s (and 512 phones) per clip, and the Waxal clips
average about 43 s. Each clip is force-aligned with the same phoneme recogniser
that annotation uses, then cut at the quietest point between two words, with a
preference for long pauses and sentence ends.

Usage: python split_clips.py --clips clips.jsonl --output DIR [--device cuda]
Input rows: {"id", "text", "speaker", "audio": path to a 24 kHz mono WAV}
Output: DIR/chunks.jsonl and DIR/wav/*.wav (24 kHz). Resumable: clips listed in
DIR/done.txt are skipped, and DIR/COMPLETE appears when every clip is done.
"""

import argparse
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import soundfile as sf

FRAME = 0.02            # aligner frame, seconds
ONSET = 0.019           # a phone starts this long before its first emission frame
LEVEL_HOP = 0.01        # energy frames for finding the quietest cut point


def word_units(text, phonemize, phone_set, unk):
    """Split text into words that carry at least one phone; a word with none
    (a lone number or symbol) is glued to the previous word."""
    units = []
    for raw in text.split():
        try:
            phones = [s for s in phonemize(raw).symbols if s in phone_set or s == unk]
        except ValueError:
            phones = []
        if phones or not units:
            units.append({"text": raw, "phones": phones})
        else:
            units[-1]["text"] += " " + raw
    if units and not units[0]["phones"]:
        if len(units) == 1:
            return []
        units[1]["text"] = units[0]["text"] + " " + units[1]["text"]
        units = units[1:]
    return units


def level_db(wave, sample_rate):
    hop = int(LEVEL_HOP * sample_rate)
    frames = len(wave) // hop
    rms = np.sqrt(np.mean(wave[:frames * hop].reshape(frames, hop) ** 2, axis=1) + 1e-10)
    return 20 * np.log10(rms)


def boundaries(units, spans, level, duration):
    """Candidate cut points between words: (time, score, after_unit_index)."""
    result, start = [], 0
    starts, ends = [], []
    for unit in units:
        count = len(unit["phones"])
        starts.append(max(0.0, spans[start][0] * FRAME - ONSET))
        ends.append((spans[start + count - 1][1] + 1) * FRAME)
        start += count
    quiet = level.max() - 35.0
    for i in range(len(units) - 1):
        a, b = ends[i], starts[i + 1]
        lo, hi = int(a / LEVEL_HOP), int(b / LEVEL_HOP)
        if hi <= lo:
            cut, pause = b, 0.0
        else:
            window = level[lo:hi]
            cut = (lo + int(np.argmin(window))) * LEVEL_HOP
            pause = float((window < quiet).sum()) * LEVEL_HOP
        score = pause
        if re.search(r"[.!?]['\"”’)]*$", units[i]["text"]):
            score += 0.3
        elif re.search(r"[,;:]['\"”’)]*$", units[i]["text"]):
            score += 0.1
        result.append((min(max(cut, 0.0), duration), score, i))
    return result


def plan(units, spans, level, duration, shortest=4.0, longest=18.0):
    """Greedy cuts: each chunk ends at the best-scoring word gap in its window.
    Returns [(start_s, end_s, first_unit, last_unit)]."""
    gaps = boundaries(units, spans, level, duration)
    chunks, start, first = [], 0.0, 0
    while duration - start > longest:
        window = [g for g in gaps if g[2] >= first and start + shortest <= g[0] <= start + longest]
        if not window:
            # no usable word gap (a very long word run): skip to the next gap after the window
            later = [g for g in gaps if g[2] >= first and g[0] > start + longest]
            if not later:
                return chunks
            start, first = later[0][0], later[0][2] + 1
            continue
        cut, _, last = max(window, key=lambda g: (g[1], g[0]))
        chunks.append((start, cut, first, last))
        start, first = cut, last + 1
    if duration - start >= 1.0 and first < len(units):
        chunks.append((start, duration, first, len(units) - 1))
    return chunks


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--clips", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--min-probability", type=float, default=0.25, help="Drop chunks the aligner is unsure of")
    args = parser.parse_args()

    os.environ.setdefault("TINY_VOCA_FRONTEND", "lug")
    from tiny_voca.align import PhoneRecognizer, token_spans
    from tiny_voca.text import PHONE_SET, UNK, phonemize, set_frontend
    set_frontend("lug")

    recognizer = PhoneRecognizer(args.device)
    (args.output / "wav").mkdir(parents=True, exist_ok=True)
    rows = [json.loads(line) for line in args.clips.read_text(encoding="utf-8").splitlines() if line.strip()]
    done_path, chunks_path = args.output / "done.txt", args.output / "chunks.jsonl"
    done = set(done_path.read_text().split()) if done_path.exists() else set()
    # Keep only chunks of fully finished clips (a clip interrupted half-way is redone).
    previous = [json.loads(l) for l in chunks_path.read_text(encoding="utf-8").splitlines() if l.strip()] if chunks_path.exists() else []
    previous = [r for r in previous if r["source"] in done]
    chunks_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in previous), encoding="utf-8")
    kept, dropped, seconds = len(previous), 0, sum(r["seconds"] for r in previous)
    print(f"resuming: {len(done)} clips already done" if done else "starting", flush=True)
    started = time.time()
    with chunks_path.open("a", encoding="utf-8") as out, done_path.open("a") as progress:
        for n, row in enumerate(rows):
            if row["id"] in done:
                continue
            wave, sample_rate = sf.read(row["audio"], dtype="float32")
            duration = len(wave) / sample_rate
            units = word_units(row["text"], phonemize, PHONE_SET, UNK)
            labels = [recognizer.label(p) for unit in units for p in unit["phones"]]
            try:
                if not labels:
                    raise ValueError("no phones")
                log_probs = recognizer.log_probs(wave, sample_rate)
                spans = token_spans(log_probs, labels, recognizer.blank)
            except ValueError:
                dropped += 1
                progress.write(row["id"] + "\n")
                progress.flush()
                continue
            probability = np.array([float(np.exp(log_probs[a:b + 1, label]).mean()) for (a, b), label in zip(spans, labels)])
            level = level_db(wave, sample_rate)
            offsets = np.cumsum([0] + [len(u["phones"]) for u in units])
            for k, (start, end, first, last) in enumerate(plan(units, spans, level, duration)):
                confidence = float(probability[offsets[first]:offsets[last + 1]].mean())
                if confidence < args.min_probability or end - start < 1.0:
                    dropped += 1
                    continue
                piece = wave[int(start * sample_rate):int(end * sample_rate)]
                name = f"{row['id']}_{k:02d}"
                path = args.output / "wav" / f"{name}.wav"
                sf.write(path, piece, sample_rate)
                text = " ".join(u["text"] for u in units[first:last + 1])
                out.write(json.dumps({"id": name, "text": text, "speaker": row["speaker"], "audio": str(path),
                                      "seconds": round(end - start, 3), "confidence": round(confidence, 3), "source": row["id"]},
                                     ensure_ascii=False) + "\n")
                kept += 1
                seconds += end - start
            out.flush()
            progress.write(row["id"] + "\n")
            progress.flush()
            if (n + 1) % 25 == 0:
                print(f"{n + 1}/{len(rows)} clips, {kept} chunks, {seconds / 60:.1f} min, "
                      f"{time.time() - started:.0f} s", flush=True)
    (args.output / "COMPLETE").write_text("ok\n")
    print(f"done: {kept} chunks ({seconds / 60:.1f} min), {dropped} dropped this run", flush=True)


if __name__ == "__main__":
    main()

Writing /kaggle/working/luganda/split_clips.py


In [6]:
# 6. Install the Luganda front end into tiny-voca and check it
subprocess.run([sys.executable, str(OUT / "apply_patch.py"), str(REPO), str(OUT / "lug.py")], check=True)
check = subprocess.run([sys.executable, "-c",
    "from tiny_voca.text import phonemize\n"
    "for s in ['Oli otya? Nkwagala nnyo, ssebo.', \"Ng'ombe za kitaawe n'abaana bajja.\"]:\n"
    "    print(s, '->', ' '.join(phonemize(s).symbols))"], capture_output=True, text=True)
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0

text.py: patched
train.py: patched
train.py: patched
annotate.py: patched
align.py: patched
Oli otya? Nkwagala nnyo, ssebo. -> <bos> ɔ l i <sp> ɔ t j ɐ ? n k w ɐ ɡ ɐ l ɐ <sp> n n j ɔ , s s ɛ b ɔ . <eos>
Ng'ombe za kitaawe n'abaana bajja. -> <bos> ŋ ɔ m b ɛ <sp> z ɐ <sp> k i t ɑː w ɛ <sp> n ɐ b ɑː n ɐ <sp> b ɐ dʒ dʒ ɐ . <eos>
 


In [7]:
# 7. Export the two speakers (train + validation splits) at 24 kHz
import numpy as np, soundfile as sf
from IPython.display import Audio, display

if not skip("export"):
    from datasets import load_dataset
    from math import gcd
    from scipy.signal import resample_poly

    ds = load_dataset("CraneAILabs/waxal-lug-clean")
    DIGITS = re.compile(r"\d")   # the front end does not read digits

    def to_24k(audio):
        x = np.asarray(audio["array"], dtype=np.float64)
        if x.ndim > 1:
            x = x.mean(axis=0 if x.shape[0] < x.shape[-1] else 1)
        sr = int(audio["sampling_rate"])
        if sr != 24000:
            g = gcd(sr, 24000)
            x = resample_poly(x, 24000 // g, sr // g)
        peak = np.abs(x).max()
        if peak > 0.99:
            x *= 0.99 / peak
        return x.astype(np.float32)

    clips, skipped = {s: [] for s in voices}, 0
    (DATA / "clips").mkdir(exist_ok=True)
    for split in ("train", "validation"):
        part = ds[split].filter(lambda s: str(s) in voices, input_columns="speaker_id")
        for row in part:
            if DIGITS.search(row["text"]):
                skipped += 1
                continue
            name = f"{split}-{row['id']}"
            wav = DATA / "clips" / f"{name}.wav"
            if not wav.exists():
                sf.write(wav, to_24k(row["audio"]), 24000)
            speaker = str(row["speaker_id"])
            clips[speaker].append({"id": name, "text": row["text"].strip(), "speaker": voices[speaker], "audio": str(wav)})
    for s, rows in clips.items():
        write_jsonl(DATA / f"clips-{s}.jsonl", rows)
        print(f"speaker {s} -> {voices[s]}: {len(rows)} clips")
    print(f"skipped {skipped} clips with digits in the text")
    tests = [t.strip() for t in ds["test"]["text"] if not DIGITS.search(t) and 30 <= len(t) <= 160][:4]
    (OUT / "test_sentences.txt").write_text("\n".join(tests) + "\n", encoding="utf-8")
    mark("export", **{voices[s]: len(rows) for s, rows in clips.items()})

for s in voices:
    first = read_jsonl(DATA / f"clips-{s}.jsonl")[0]
    x, sr = sf.read(first["audio"])
    print(f"speaker {s} ({voices[s]}):")
    display(Audio(x[:8 * sr], rate=sr))

README.md:   0%|          | 0.00/3.02k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  574MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  225MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  310MB            

... (38 outputs left out)

Filter:   0%|          | 0/1608 [00:00<?, ? examples/s]

Filter:   0%|          | 0/211 [00:00<?, ? examples/s]

speaker 5 -> F2: 228 clips
speaker 1 -> M2: 232 clips
skipped 1 clips with digits in the text
✓ export finished: {'F2': 228, 'M2': 232}
speaker 5 (F2):


speaker 1 (M2):


In [8]:
# 8. Cut long clips into 4-18 s chunks at word pauses: one speaker per GPU (resumes clip by clip)
if not skip("split"):
    jobs = {}
    for gpu, s in enumerate(voices):
        out = DATA / f"chunks-{s}"
        if (out / "COMPLETE").exists():
            continue
        jobs[f"split-{s}"] = start(f"split-{s}", [sys.executable, str(OUT / "split_clips.py"),
                                                  "--clips", str(DATA / f"clips-{s}.jsonl"), "--output", str(out)], gpu)
    watch(jobs)
    minutes = {voices[s]: round(sum(r["seconds"] for r in read_jsonl(DATA / f"chunks-{s}" / "chunks.jsonl")) / 60)
               for s in voices}
    mark("split", minutes=minutes)

[   3.7 min] split-5: working (prints when done)
[   3.7 min] split-1: working (prints when done)
[   4.7 min] split-5: 175/228 clips, 346 chunks, 57.9 min, 29 s
[   4.7 min] split-1: 175/232 clips, 383 chunks, 69.1 min, 36 s
[   5.7 min] split-5: 200/228 clips, 700 chunks, 126.2 min, 74 s
[   5.7 min] split-1: 200/232 clips, 624 chunks, 116.0 min, 62 s
split-5: done: 921 chunks (169.4 min), 4 dropped this run
split-1: done: 1007 chunks (192.0 min), 0 dropped this run
✓ split finished: {'minutes': {'F2': 169, 'M2': 192}}


In [9]:
# 9. Training manifests and acoustic features (sharded across CPU cores; each shard resumes)
if not skip("features"):
    rows = {"train": [], "validation": []}
    if not (DATA / "train.jsonl").exists():   # written once, so a resumed run sees identical inputs
        for s, voice in voices.items():
            chunks = sorted(read_jsonl(DATA / f"chunks-{s}" / "chunks.jsonl"), key=lambda r: (-r["confidence"], r["id"]))
            total = 0.0
            for r in chunks:
                if total >= MINUTES_PER_VOICE * 60:
                    break
                total += r["seconds"]
                # split by text so the same sentence never lands in both splits
                bucket = int(hashlib.sha256(r["text"].strip().encode()).hexdigest()[:8], 16) % 25
                rows["validation" if bucket == 0 else "train"].append(
                    {"id": r["id"], "text": r["text"], "voice_id": voice, "audio": r["audio"]})
            print(f"{voice}: {total / 60:.0f} min kept from {len(chunks)} chunks")
        cores = os.cpu_count() or 1
        for split, items in rows.items():
            parts = max(1, min(cores, len(items) // 20)) if split == "train" else 1
            for k in range(parts):
                write_jsonl(DATA / f"{split}-{k}.jsonl", items[k::parts])
            write_jsonl(DATA / f"{split}.jsonl", items)
            print(split, len(items), "chunks in", parts, "shard(s)")

    jobs = {}
    for part in sorted(DATA.glob("train-*.jsonl")) + sorted(DATA.glob("validation-*.jsonl")):
        shard = FEATURES / "shards" / part.stem
        meta = shard / f"{part.stem.rsplit('-', 1)[0]}.preparation.json"
        if meta.exists() and json.loads(meta.read_text()).get("complete"):
            continue
        jobs[f"prepare-{part.stem}"] = start(f"prepare-{part.stem}", [sys.executable, "-m", "tiny_voca.prepare",
            "--inputs", str(part), "--split", part.stem.rsplit("-", 1)[0], "--output", str(shard), "--resume"],
            single_thread=True)
    print(f"{len(jobs)} feature jobs to run on {os.cpu_count()} CPU cores")
    watch(jobs, every=120)

    # One manifest per split, pointing into the shard folders (nothing is moved, so shards stay resumable)
    counts = {}
    for split in ("train", "validation"):
        merged = []
        for shard in sorted((FEATURES / "shards").glob(f"{split}-*")):
            for r in read_jsonl(shard / f"{split}.jsonl"):
                r["path"] = f"shards/{shard.name}/{r['path']}"
                merged.append(r)
            shutil.copy(shard / "features.json", FEATURES / "features.json")
        write_jsonl(FEATURES / f"{split}.jsonl", merged)
        (FEATURES / f"{split}.preparation.json").write_text(json.dumps({"complete": True, "accepted": len(merged)}))
        counts[split] = len(merged)
    mark("features", **counts)

F2: 100 min kept from 921 chunks
M2: 100 min kept from 1007 chunks
train 1048 chunks in 4 shard(s)
validation 39 chunks in 1 shard(s)
5 feature jobs to run on 4 CPU cores
[   6.7 min] prepare-train-0: working (prints when done)
[   6.7 min] prepare-train-1: working (prints when done)
[   6.7 min] prepare-train-2: working (prints when done)
[   6.7 min] prepare-train-3: working (prints when done)
[   6.7 min] prepare-validation-0: working (prints when done)
[   8.7 min] prepare-train-0: working (prints when done)
[   8.7 min] prepare-train-1: working (prints when done)
[   8.7 min] prepare-train-2: working (prints when done)
[   8.7 min] prepare-train-3: working (prints when done)
[   8.7 min] prepare-validation-0: working (prints when done)
[  10.7 min] prepare-train-0: working (prints when done)
[  10.7 min] prepare-train-1: working (prints when done)
[  10.7 min] prepare-train-2: working (prints when done)
[  10.7 min] prepare-train-3: working (prints when done)
[  10.7 min] prepare-

In [10]:
# 10. Phoneme durations by forced alignment (resumes from its journal)
from collections import Counter
if not skip("annotate"):
    command = [sys.executable, "-m", "tiny_voca.annotate",
               "--features", str(FEATURES), "--output", str(ANN), "--dataset-revision", "CraneAILabs/waxal-lug-clean",
               "--splits", "train", "validation", "--min-token-probability", "0.35", "--max-weak-phone-fraction", "0.5",
               "--max-phone-frames", "90"]
    if (ANN / "annotations.json").exists():
        command.append("--resume")
    watch({"annotate": start("annotate", command, 0)}, every=60)
    assert json.loads((ANN / "annotations.json").read_text())["complete"], "annotation run is not marked complete"
    report = {}
    for split in ("train", "validation"):
        ok, bad = read_jsonl(ANN / f"{split}.jsonl"), read_jsonl(ANN / f"{split}.rejected.jsonl")
        minutes = Counter()
        for r in ok:
            minutes[str(r.get("speaker"))] += r.get("frames", 0) * 256 / 24000 / 60
        report[split] = {"accepted": len(ok), "rejected": len(bad), "minutes": {k: round(v) for k, v in minutes.items()}}
        print(f"{split}: {report[split]}")
        print("  rejection reasons:", Counter(reason for r in bad for reason in r["reasons"]).most_common())
    mark("annotate", **report)

[  36.7 min] annotate: working (prints when done)
[  37.7 min] annotate: {"elapsed": 36.2, "message": "aligned train 500/1048"}
annotate: {"elapsed": 68.0, "message": {"complete": true, "splits": {"train": {"accepted": 1040, "rejected": 8, "accepted_by_voice": {"F2": 523, "M2": 517}, "accepted_hours": 3.177, "rejection_reasons": {"overlong_phone": 8, "implausible_rate": 3}}, "validation": {"accepted": 39, "rejected": 0, "accepted_by_v
train: {'accepted': 1040, 'rejected': 8, 'minutes': {'F2': 95, 'M2': 96}}
  rejection reasons: [('overlong_phone', 8), ('implausible_rate', 3)]
validation: {'accepted': 39, 'rejected': 0, 'minutes': {'F2': 4, 'M2': 4}}
  rejection reasons: []
✓ annotate finished: {'train': {'accepted': 1040, 'rejected': 8, 'minutes': {'F2': 95, 'M2': 96}}, 'validation': {'accepted': 39, 'rejected': 0, 'minutes': {'F2': 4, 'M2': 4}}}


In [11]:
# 11. Train ONE model on both T4s (data-parallel via torchrun); resumes from run/last.pt
import torch
TRAIN_SETTINGS = LOGS / "train-settings.json"   # outside run/: the trainer needs an empty folder to start

def saved_step(path):
    return torch.load(path, map_location="cpu", weights_only=True)["step"] if path.exists() else 0

def launch(batch_frames, seconds, resume):
    if not resume:
        shutil.rmtree(RUN, ignore_errors=True)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", "2",
           "-m", "tiny_voca.train",
           "--train", str(ANN / "train.jsonl"), "--validation", str(ANN / "validation.jsonl"),
           "--output", str(RUN), "--init-weights", str(WEIGHTS),
           "--steps", str(STEPS), "--max-seconds", str(int(seconds)),
           "--learning-rate", str(LEARNING_RATE), "--warmup", "300",
           "--gan-start", "1000", "--gan-ramp", "1000",
           "--teacher-hold", "500", "--teacher-ramp", "1500", "--teacher-final", "0",
           "--batch-frames", str(batch_frames), "--save-every", "500", "--keep-every", "1000", "--log-every", "50"]
    if resume:
        cmd += ["--resume", str(RUN / "last.pt")]
    log = LOGS / "train.log"
    with open(log, "a") as stream:
        stream.write(f"\n=== {'resume' if resume else 'start'} at {time.ctime()} (batch frames {batch_frames}) ===\n")
    proc = subprocess.Popen(cmd, env=dict(os.environ), stdout=open(log, "a"), stderr=subprocess.STDOUT)
    if not resume:
        TRAIN_SETTINGS.write_text(json.dumps({"batch_frames": batch_frames}))
    return proc, log

def progress(text):
    text = text.rsplit("\n=== ", 1)[-1]   # this launch only
    for line in reversed(text.splitlines()):
        if line.startswith("{") and '"steps_per_second"' in line:
            try:
                return json.loads(line)
            except json.JSONDecodeError:
                pass
    return None

if not skip("train"):
    step = saved_step(RUN / "last.pt")
    if step >= STEPS:
        mark("train", step=step)
    else:
        resume = step > 0
        batch_frames = json.loads(TRAIN_SETTINGS.read_text())["batch_frames"] if resume else BATCH_FRAMES
        print(f"resuming at step {step}" if resume else "starting from the English weights")
        began = time.time()
        proc, log = launch(batch_frames, MAX_TRAIN_SECONDS, resume)
        while True:
            time.sleep(120)
            code, text = proc.poll(), log.read_text(errors="ignore")
            r = progress(text)
            if r:
                left = (STEPS - r["step"]) / max(r["steps_per_second"], 1e-9) / 60
                print(f"[{(time.time() - began) / 60:5.1f} min] step {r['step']}/{STEPS}  {r['steps_per_second']:.2f} steps/s  "
                      f"loss {r['loss']:.3f}  ~{left:.0f} min left", flush=True)
            if code is None:
                continue
            last = text.rsplit("\n=== ", 1)[-1]
            if code != 0 and "out of memory" in last and batch_frames > 6000 and saved_step(RUN / "last.pt") == 0:
                batch_frames = 6000
                print("A T4 ran out of memory before the first save: restarting with --batch-frames 6000")
                proc, log = launch(batch_frames, MAX_TRAIN_SECONDS - (time.time() - began), False)
                continue
            if code != 0:
                print(last[-5000:])
                raise RuntimeError(f"training failed with exit code {code}; rerun this cell to resume from the last save")
            break
        step = saved_step(RUN / "last.pt")
        if step >= STEPS:
            mark("train", step=step, minutes=round((time.time() - began) / 60))
        else:
            print(f"Stopped at step {step}/{STEPS} by the time cap: run this cell again (or the next session) to continue.")
print("kept checkpoints:", sorted(p.name for p in RUN.glob("step-*.pt")))

starting from the English weights
[  2.0 min] step 300/4000  2.84 steps/s  loss 11.963  ~22 min left
[  4.0 min] step 600/4000  2.76 steps/s  loss 10.631  ~21 min left
[  6.0 min] step 950/4000  2.78 steps/s  loss 10.430  ~18 min left
[  8.0 min] step 1100/4000  0.97 steps/s  loss 21.091  ~50 min left
[ 10.0 min] step 1200/4000  0.94 steps/s  loss 29.966  ~50 min left
[ 12.0 min] step 1300/4000  0.95 steps/s  loss 26.779  ~48 min left
[ 14.0 min] step 1400/4000  0.93 steps/s  loss 22.062  ~47 min left
[ 16.0 min] step 1550/4000  0.98 steps/s  loss 20.162  ~42 min left
[ 18.0 min] step 1650/4000  0.93 steps/s  loss 22.192  ~42 min left
[ 20.0 min] step 1750/4000  0.95 steps/s  loss 22.099  ~39 min left
[ 22.0 min] step 1900/4000  0.95 steps/s  loss 25.251  ~37 min left
[ 24.0 min] step 2000/4000  0.97 steps/s  loss 26.487  ~34 min left
[ 26.0 min] step 2100/4000  0.94 steps/s  loss 25.833  ~34 min left
[ 28.0 min] step 2200/4000  0.95 steps/s  loss 25.738  ~31 min left
[ 30.0 min] step 

In [12]:
# 12. Listen to the last checkpoints (both voices, held-out sentences) and pick one
from tiny_voca.text import set_frontend
set_frontend("lug")
from tiny_voca.infer import load_model, speak_text

CKPT = OUT / "checkpoints"
CKPT.mkdir(exist_ok=True)
candidates = {}
for state in sorted(RUN.glob("step-*.pt"))[-3:]:          # e.g. steps 2000, 3000, 4000
    target = CKPT / f"{state.stem}.pt"
    if not target.exists():
        subprocess.run([sys.executable, str(REPO / "scripts" / "export_checkpoint.py"),
                        "--checkpoint", str(state), "--output", str(target)], check=True)
    candidates[state.stem] = target
final_step = saved_step(RUN / "inference.pt")
if f"step-{final_step:08d}" not in candidates:             # stopped between kept steps
    candidates[f"step-{final_step:08d}"] = RUN / "inference.pt"

sentences = [s for s in (OUT / "test_sentences.txt").read_text(encoding="utf-8").splitlines() if s.strip()]
for name, checkpoint in candidates.items():
    model = load_model(checkpoint)
    for voice in ("F2", "M2"):
        for i, sentence in enumerate(sentences[:2]):
            path = OUT / "samples" / name / f"{voice}-{i}.wav"
            if not path.exists():
                wave, _ = speak_text(model, sentence, voice)
                path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(path, wave.detach().cpu().numpy(), model.config.sample_rate)
            print(f"{name} | {voice} | {sentence}")
            display(Audio(str(path)))

exported step 2000 with 4,537,291 parameters to /kaggle/working/luganda/checkpoints/step-00002000.pt
exported step 3000 with 4,537,291 parameters to /kaggle/working/luganda/checkpoints/step-00003000.pt
exported step 4000 with 4,537,291 parameters to /kaggle/working/luganda/checkpoints/step-00004000.pt
step-00002000 | F2 | Ennaku zino akawunga ye baana baliwo mu Uganda yonna, buli Maka okasangamu.


step-00002000 | F2 | Obummonde obuzungu buwooma nnyo naye bulumya enjala, olya n'obeera ng’atalidde.


step-00002000 | M2 | Ennaku zino akawunga ye baana baliwo mu Uganda yonna, buli Maka okasangamu.


step-00002000 | M2 | Obummonde obuzungu buwooma nnyo naye bulumya enjala, olya n'obeera ng’atalidde.


step-00003000 | F2 | Ennaku zino akawunga ye baana baliwo mu Uganda yonna, buli Maka okasangamu.


step-00003000 | F2 | Obummonde obuzungu buwooma nnyo naye bulumya enjala, olya n'obeera ng’atalidde.


... (2 outputs left out)

step-00004000 | F2 | Ennaku zino akawunga ye baana baliwo mu Uganda yonna, buli Maka okasangamu.


step-00004000 | F2 | Obummonde obuzungu buwooma nnyo naye bulumya enjala, olya n'obeera ng’atalidde.


step-00004000 | M2 | Ennaku zino akawunga ye baana baliwo mu Uganda yonna, buli Maka okasangamu.


step-00004000 | M2 | Obummonde obuzungu buwooma nnyo naye bulumya enjala, olya n'obeera ng’atalidde.


In [13]:
# 13. Package the chosen checkpoint in /kaggle/working/luganda/model (PyTorch + ONNX)
BEST = None   # after listening, e.g. "step-00003000"; None keeps the last checkpoint
best = BEST or list(candidates)[-1]
final = OUT / "model"
final.mkdir(exist_ok=True)
shutil.copy(candidates[best], final / "tiny-voca-lug.pt")
for name in ("lug.py", "apply_patch.py"):
    shutil.copy(OUT / name, final / name)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=False)
onnx = subprocess.run([sys.executable, "-m", "tiny_voca.export", "--checkpoint", str(final / "tiny-voca-lug.pt"),
                       "--output", str(final / "model.onnx")], capture_output=True, text=True)
print(onnx.stdout[-1500:], onnx.stderr[-1500:])

(final / "README.md").write_text(f'''# tiny-voca Luganda

A 4.5M-parameter Luganda text-to-speech model with two voices: F2 (female) and M2 (male).
Fine-tuned from [n4ze3m/tiny-voca](https://huggingface.co/n4ze3m/tiny-voca) (Apache-2.0) on
[CraneAILabs/waxal-lug-clean](https://huggingface.co/datasets/CraneAILabs/waxal-lug-clean)
(the WaxalNLP Luganda recordings, CC BY-SA 4.0): speaker {FEMALE} as F2 and speaker {MALE} as M2. Checkpoint: {best}.
The weights are released under CC BY-SA 4.0, like the recordings.

## Use

```bash
git clone https://github.com/n4ze3m/tiny-voca && pip install -e tiny-voca
python apply_patch.py tiny-voca lug.py
TINY_VOCA_FRONTEND=lug python -m tiny_voca.infer --checkpoint tiny-voca-lug.pt --voice F2 \\
  --text "Oli otya? Nkwagala nnyo." --output out.wav
```

## Limits

- Pronunciation comes from spelling rules (`lug.py`). Tone is not written in Luganda and is not modelled.
- Digits are not read: write numbers out in words.
''', encoding="utf-8")
mark("package", checkpoint=best)
print("model folder:", sorted(p.name for p in final.iterdir()))

# Optional: push to Hugging Face (add an HF_TOKEN secret under Add-ons -> Secrets first)
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import HfApi
# api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
# api.create_repo("YOUR_NAME/tiny-voca-lug", exist_ok=True)
# api.upload_folder(folder_path=str(final), repo_id="YOUR_NAME/tiny-voca-lug")

wrote /kaggle/working/luganda/model/model.onnx (21.4 MB) and model.json
 dynamic_axes=axes,
/kaggle/working/tiny-voca/src/tiny_voca/model.py:60: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  q, k, v = self.qkv(x).view(b, t, 3, self.heads, c // self.heads).permute(2, 0, 3, 1, 4)
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:714: UserWarning: Constant folding - Only steps=1 can b